In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
import warnings

from SharedModules import input_dir, output_dir, model_dir
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

cv = KFold(n_splits=10, shuffle=True, random_state=42)
scaler = StandardScaler()

_train = pd.read_csv(input_dir + "train.csv", index_col=0)
_test = pd.read_csv(input_dir + "test.csv", index_col=0)

target = "Heart Disease"

X = _train.drop(target, axis= 1)
y = _train[target].map({"Absence": 0, "Presence": 1})
X_test = _test

In [3]:
keras = tf.keras
layers = keras.layers


def build_model(n_features):
    model = keras.Sequential(
        [
            layers.Dense(32, activation="relu", input_shape=[n_features]),
            layers.BatchNormalization(),
            layers.Dropout(0.2),
            layers.Dense(16, activation="relu"),
            layers.BatchNormalization(),
            layers.Dropout(0.2),
            layers.Dense(1, activation="sigmoid"),
        ]
    )

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["AUC"],
    )

    return model


early_stopping = keras.callbacks.EarlyStopping(
    patience=10,
    min_delta=0.001,
    restore_best_weights=True,
)

oof = np.zeros(len(X))
oof_scores = []

for fold, (idx_tr, idx_val) in enumerate(cv.split(X, y)):
    X_tr, X_val = X.iloc[idx_tr], X.iloc[idx_val]
    y_tr, y_val = y.iloc[idx_tr], y.iloc[idx_val]

    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)

    model = build_model(X.shape[1])

    model.fit(
        X_tr_scaled,
        y_tr,
        validation_data=(X_val_scaled, y_val),
        batch_size=512,
        epochs=1000,
        callbacks=[early_stopping],
        verbose=0,
    )
    y_pred = model.predict(X_val_scaled).ravel()
    score = roc_auc_score(y_val, y_pred)

    oof[idx_val] = y_pred
    oof_scores.append(score)

    print(f"fold {fold + 1} score: {score:.4f}")

print(f"Average Score: {np.mean(oof_scores): .4f}")

df_oof = pd.DataFrame(data=oof, index=X.index, columns=["Keras"])
df_oof.to_csv(model_dir + "Keras/oof.csv")

1969/1969 ━━━━━━━━━━━━━━━━━━━━ 1s 406us/step
fold 1 score: 0.9530
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 1s 396us/step
fold 2 score: 0.9510
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 1s 401us/step
fold 3 score: 0.9516
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 1s 397us/step
fold 4 score: 0.9523
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 1s 393us/step
fold 5 score: 0.9507
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 1s 539us/step
fold 6 score: 0.9550
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 1s 401us/step
fold 7 score: 0.9504
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 1s 407us/step
fold 8 score: 0.9513
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 1s 411us/step
fold 9 score: 0.9523
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 1s 408us/step
fold 10 score: 0.9517
Average Score:  0.9519


In [5]:
final_model = build_model(X.shape[1])

X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

final_model.fit(
    X_scaled,
    y,
    validation_split=0.1,  # hold out 10% internally
    batch_size=512,
    epochs=1000,
    callbacks=[early_stopping],
    verbose=0,
)

sub = pd.DataFrame(data=final_model.predict(X_test_scaled).squeeze(), index=X_test.index, columns=["Keras"])
sub.to_csv(output_dir + "submission_keras.csv")

8438/8438 ━━━━━━━━━━━━━━━━━━━━ 3s 385us/step
